[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/wing/wing.ipynb)

# Potential Flow past a Joukowski Wing

This example computes the potential flow past a Joukowski wing and its aerodynamic lift. In an incompressible, inviscid fluid, parcels neither compress nor acquire vorticity, so the velocity's derivative vanishes at every point outside the body. All vorticity is bound inside the wing itself, appearing in the surrounding fluid as a constant circulation around every closed loop that encloses the wing. The flow is constructed around a circular cylinder and carried onto the wing by Joukowski's angle-preserving map.

Points and the velocities that move them are modelled as antivectors, while forces on the wing are modelled as vectors that pair with displacements via `force & displacement` to give mechanical work. Multiplying the velocity's gradient map by `Point` and contracting gives the velocity's geometric derivative `(Point * gradient(Point)).contract(1, 2)`, whose scalar divergence and bivector vorticity both vanish. Stagnation at the trailing edge sets the bivector Kutta circulation `4 * pi * (stream ^ edge)`, joining that circulation with the stream gives the lift `-density * (swirl & stream)`, and the Joukowski map `positions + critical * positions.inverse() * critical` carries the cylinder's potential gradient to the wing through the inverse of its Jacobian `Point - ((critical * positions.inverse()) >> Point)`.

In complex-variable potential-flow notation, the plane is identified with $z = x + iy$ in a chosen coordinate frame, the cylinder potential $W(\zeta) = U e^{-i\alpha}(\zeta - \zeta_0) + U e^{i\alpha} R^2/(\zeta - \zeta_0) + \frac{i\Gamma}{2\pi}\ln(\zeta - \zeta_0)$ with clockwise circulation $\Gamma = 4\pi U R \sin(\alpha + \beta)$ is mapped by $z = \zeta + c^2/\zeta$, the velocity components $(u, v)$ are recovered from the conjugate derivative $u - iv = (dW/d\zeta)/(1 - c^2/\zeta^2)$, and the Kutta–Joukowski force is written as $F_x + i F_y = i\rho U e^{i\alpha}\Gamma$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA2D as ga
from examples.util.animation import save_figure, show_states
from numga.topology import Topology
from examples.fluids.wing import render

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                  # Scalar+
# Points, displacements, and velocities of points are antivectors with intrinsic orientation.
Point = ga.gatype.antivector()                               # Antivector-
# A force on the wing pairs with a displacement of the wing via `force & displacement` to give work.
Force = ga.gatype.vector()                                   # Vector+
Bivector = ga.gatype.bivector()                              # Bivector+
Rotor = ga.gatype.rotor()                                    # Rotor+
VelocityGradient = ga.gatype((Point, Point))                 # Point- <- Point-
PotentialGradient = ga.gatype((Scalar, Point))               # Scalar+ <- Point-

## Cylinder geometry and Kutta circulation

The wing begins as a circular cylinder whose centre is offset behind and above the origin before the map. Folding the cylinder's rear point into a sharp trailing edge requires the flow to stagnate there so that the mapped stream leaves the cusp smoothly, a requirement called the Kutta condition. That stagnation fixes a unique bivector circulation `4 * pi * (stream ^ edge)` from the free stream and the radial offset `edge` to the trailing edge. Joining the circulation with the stream gives the Kutta–Joukowski lift `-density * (swirl & stream)`, directed across the stream toward the faster, lower-pressure side.

In scalar airfoil notation, the circulation and lift magnitude read as $\Gamma = 4\pi U R \sin(\alpha + \beta)$ and $L = \rho U \Gamma$ for a cylinder of radius $R$ whose trailing edge lies at angle $\beta$ below its centre.

In [ ]:
# Cylinder offset setting thickness and camber, free-stream speed and density, and angle of attack.
thickness, camber, chord_scale = 0.1, 0.08, 1.0
speed, density = 1.0, 1.0
attack = np.deg2rad(8.0)

In [ ]:
def wing_geometry(attack_angle: float) -> tuple[Point, Point]:
    """Cylinder centre and trailing-edge point for the wing pitched nose up by `attack_angle`."""
    rotor = (mv.xy * (attack_angle / 2)).exp()                                 # [] Rotor+
    centre = rotor >> (-thickness * mv.x + camber * mv.y).dual()               # [] Point-
    trailing_edge = chord_scale * (rotor >> mv.x.dual())                       # [] Point-
    return centre, trailing_edge


centre, trailing_edge = wing_geometry(attack)                                  # [] Point-, Point-
stream = speed * mv.x.dual()                                                   # [] Point-
# Radial offset from the cylinder centre to the trailing-edge stagnation point.
edge = trailing_edge - centre                                                  # [] Point-
swirl = 4 * np.pi * (stream ^ edge)                                            # [] Bivector+
lift = -density * (swirl & stream)                                             # [] Force+

## Flow past the cylinder

Around the cylinder, the velocity combines the uniform stream, minus its reflection `radius_squared * (inverse * stream * inverse)` in the cylinder wall so that no fluid crosses the surface, and the azimuthal swirl from the circulation. Because a small step changes the position inverse by `-(inverse * step * inverse)`, differentiating each velocity term gives the linear velocity gradient map directly, and contracting `(Point * gradient(Point)).contract(1, 2)` confirms that both divergence and vorticity vanish throughout the exterior.

Along any step, the geometric product `velocity * step` splits into the potential increment `velocity | step` along the flow and the bivector flux `velocity ^ step` across the step, which integrates to the stream function.

In [ ]:
# Concentric sample rings around the cylinder, angles per ring, and outer radius in cylinder radii.
n_rings, n_angles, outer_distance = 140, 360, 6.0

In [ ]:
def cylinder_rings(centre: Point, trailing_edge: Point, n_rings: int, n_angles: int, outer_distance: float) -> Point:
    """Concentric sample rings around `centre` out to `outer_distance` radii, offset half a step from `trailing_edge`."""
    edge = trailing_edge - centre                                              # [] Point-
    growth = np.exp(np.linspace(0, np.log(outer_distance), n_rings))
    turns = (mv.xy * (-(np.arange(n_angles) + 0.5) / n_angles * np.pi)).exp()  # [n_angles] Rotor+
    return centre + (turns >> edge)[None, :] * growth[:, None]                 # [n_rings, n_angles] Point-


def cylinder_flow(centre: Point, trailing_edge: Point, swirl: Bivector, stream: Point,
                  positions: Point) -> tuple[Point, VelocityGradient, Bivector]:
    """The velocity, velocity gradient map, and stream function past the cylinder through `trailing_edge`."""
    offsets = positions - centre                                               # [...] Point-
    edge = trailing_edge - centre                                              # [] Point-
    radius_squared = edge.squared()                                            # [] Scalar+
    inverse = offsets.inverse()                                                # [...] Point-
    # The uniform stream, minus its reflection in the cylinder, plus the circulation swirl around it.
    velocity = stream - radius_squared * (inverse * stream * inverse) - swirl * inverse / (2 * np.pi)   # [...] Point-
    # A small step changes the position inverse by minus the step sandwiched between two inverses.
    inverse_gradient = -(inverse * Point * inverse)                            # [...] Point- <- Point-
    gradient = (                                                               # [...] Point- <- Point-
        -radius_squared * (inverse_gradient * stream * inverse + inverse * stream * inverse_gradient)
        - swirl * inverse_gradient / (2 * np.pi)
    )
    # Stream function: net flow crossing from the centre's streamline, its image, and the logarithmic swirl.
    flux = (stream ^ offsets) + radius_squared * (inverse ^ stream) - swirl * offsets.squared().log() / (4 * np.pi)   # [...] Bivector+
    return velocity, gradient, flux


positions = cylinder_rings(centre, trailing_edge, n_rings, n_angles, outer_distance)   # [n_rings, n_angles] Point-
cylinder_velocity, cylinder_gradient, cylinder_flux = cylinder_flow(centre, trailing_edge, swirl, stream, positions)
# Geometric derivative of the velocity: scalar divergence plus bivector vorticity.
cylinder_derivatives = (Point * cylinder_gradient(Point)).contract(1, 2)       # [n_rings, n_angles] Even+

## Bipolar circles and the Joukowski map

The Joukowski map `positions + critical * positions.inverse() * critical` adds to each point its inverse reflected in the line along `critical` and scaled by `critical.squared()`. Its Jacobian `Point - ((critical * positions.inverse()) >> Point)` turns and scales every step, preserving crossing angles everywhere except at `critical` and `-critical`, where the Jacobian vanishes, angles double, and the two points land at `2 * critical` and `-2 * critical`.

Circles through both critical points map to circles through their images, while Apollonius circles of constant distance ratio to the two critical points map to circles around the images with the ratio squared. Because the cylinder passes through `critical` and encloses `-critical`, the doubled angle at `critical` folds its smooth boundary into a sharp trailing-edge cusp.

In complex-variable notation, the map reads as $z = \zeta + c^2/\zeta$, the two circle families form the bipolar coordinates of $\pm c$, and angle doubling at the critical points follows from $\frac{z - 2c}{z + 2c} = \left(\frac{\zeta - c}{\zeta + c}\right)^2$.

In [ ]:
# Bipolar circle offsets across the chord, Apollonius distance ratios, and sample count per circle.
through_offsets = np.array([-1.5, -0.6, 0.0, 0.6, 1.5])
distance_ratios = np.array([0.2, 0.45, 0.7])
n_outline_points = 399

In [ ]:
def joukowski_image(critical_point: Point, sample_points: Point) -> Point:
    """Image points under the Joukowski map with critical points at `critical_point` and `-critical_point`."""
    return sample_points + critical_point * sample_points.inverse() * critical_point   # [...] Point-


outline_turns = (mv.xy * np.linspace(0, np.pi, n_outline_points, endpoint=False)).exp()   # [n_outline_points] Rotor+
chord_unit = trailing_edge / chord_scale                                       # [] Point-
# Circles through both critical points have centres on the perpendicular bisector of the segment between them.
quarter_turn = (mv.xy * (-np.pi / 4)).exp()                                    # [] Rotor+
through_centres = through_offsets * (quarter_turn >> trailing_edge)            # [5] Point-
through_radii = (trailing_edge.squared() + through_centres.squared()).square_root()   # [5] Scalar+
through_circles = through_centres[:, None] + through_radii[:, None] * (outline_turns >> chord_unit)[None, :]   # [5, n_outline_points] Point-
# Apollonius circles around each critical point hold a fixed ratio of distances to the two critical points.
all_ratios = np.concatenate([distance_ratios, 1 / distance_ratios])
around_centres = trailing_edge * ((1 + all_ratios**2) / (1 - all_ratios**2))   # [6] Point-
around_radii = trailing_edge.norm() * (2 * all_ratios / np.abs(1 - all_ratios**2))   # [6] Scalar+
around_circles = around_centres[:, None] + around_radii[:, None] * (outline_turns >> chord_unit)[None, :]   # [6, n_outline_points] Point-
bipolar_grid = concatenate([through_circles, around_circles])                  # [11, n_outline_points] Point-
cylinder_outline = centre + (outline_turns >> edge)                            # [n_outline_points] Point-
save_figure(
    render.bipolar(
        bipolar_grid, joukowski_image(trailing_edge, bipolar_grid),
        cylinder_outline, joukowski_image(trailing_edge, cylinder_outline), trailing_edge,
    ),
    "wing_map",
);

Grey curves show the two orthogonal families of bipolar circles on the left and their images on the right, with the circle centred at the origin flattening onto the slit between the two yellow critical-point markers. The dark loop is the offset cylinder on the left and the cambered Joukowski wing with its cusped trailing edge on the right.

## Carrying the flow to the wing

Assigning each image point the velocity potential and stream function of its preimage carries the cylinder's flow onto the wing. Because the Jacobian turns and scales steps everywhere outside the wing, the carried velocity has the cylinder's divergence and vorticity divided by the local scale squared, so both stay identically zero. Constant-flux curves map to constant-flux curves, keeping the wing boundary a streamline while leaving the uniform stream undisturbed at infinity.

Pulling a step in the wing plane back through `jacobian.inverse()` composes the cylinder's potential gradient `velocity | Point` into the wing's potential gradient, whose contraction with `Point` gives the velocity around the wing.

In [ ]:
def wing_flow(centre: Point, trailing_edge: Point, swirl: Bivector, stream: Point,
              positions: Point, critical_point: Point) -> tuple[Point, Point, PotentialGradient, Bivector]:
    """Image points, wing velocity, potential gradient map, and stream function under the Joukowski map."""
    velocity, _, flux = cylinder_flow(centre, trailing_edge, swirl, stream, positions)
    # How the map moves a small step: the step minus the step turned and scaled by `critical_point * positions.inverse()`.
    jacobian = Point - ((critical_point * positions.inverse()) >> Point)       # [...] Point- <- Point-
    # Pulling a step at the image back through `jacobian.inverse()` gives the potential gradient at the wing.
    potential_gradient = (velocity | Point)(jacobian.inverse())                # [...] Scalar+ <- Point-
    mapped_velocity = (Point * potential_gradient(Point)).contract(1, 2)       # [...] Point-
    return joukowski_image(critical_point, positions), mapped_velocity, potential_gradient, flux


wing_points, wing_velocity, wing_potential_gradient, wing_flux = wing_flow(
    centre, trailing_edge, swirl, stream, positions, trailing_edge,
)                                                                              # [n_rings, n_angles] Point-, Point-, (Scalar+ <- Point-), Bivector+
save_figure(render.figure(wing_points, wing_velocity, wing_flux, lift, speed), "wing_flow");

Red shading marks low Bernoulli pressure where the flow accelerates over the cambered upper surface, blue marks high pressure near the leading-edge stagnation point, white marks free-stream pressure, dark curves trace evenly spaced streamlines, and the green arrow shows the Kutta–Joukowski lift.

Separating the critical points continuously from the origin out to `trailing_edge` deforms the circular cylinder into the wing while holding the bound circulation fixed. Rendering the 92-frame morph cycle takes about eight seconds.

In [ ]:
# Sample grid resolution and frame count for the morphing animation.
n_drawn_rings, n_drawn_angles = 35, 90
n_morph_frames, frame_duration_ms = 92, 40

In [ ]:
def morph_states(n_frames: int) -> Iterator[tuple[Point, Point, Bivector, Force, Point, Point]]:
    """Yield flow states over a smooth cycle as the critical points separate to the trailing edge and return."""
    drawn_positions = cylinder_rings(centre, trailing_edge, n_drawn_rings, n_drawn_angles, outer_distance)   # [n_drawn_rings, n_drawn_angles] Point-
    fractions = (1 - np.cos(np.linspace(0, 2 * np.pi, n_frames, endpoint=False))) / 2
    for fraction in fractions:
        critical_point = fraction * trailing_edge                              # [] Point-
        critical_images = 2 * stack([critical_point, -critical_point])         # [2] Point-
        points, velocity, _, flux = wing_flow(centre, trailing_edge, swirl, stream, drawn_positions, critical_point)
        yield points, velocity, flux, lift, critical_images, centre


morphed_states = show_states(
    morph_states(n_morph_frames), render.animate_marked, speed,
    name="wing_morph", duration_ms=frame_duration_ms, n_frames=n_morph_frames,
);

Yellow markers track the critical-point images `2 * critical_point` and `-2 * critical_point`, starting together at the origin while the white cylinder centre sits behind and above them by `thickness` and `camber`. While the critical points remain inside the circle the body stays smooth, and at the midpoint of the cycle `critical_point` reaches the boundary to form the trailing-edge cusp.

Pitching the wing turns the cylinder centre and trailing edge together with one rotor, updating `4 * pi * (stream ^ edge)` at each angle of attack so that stagnation stays pinned to the cusp as lift rises and falls. Rendering the 120-frame pitching cycle takes about ten seconds.

In [ ]:
# Angle-of-attack range and frame count for one pitching cycle.
lowest_attack, highest_attack = np.deg2rad(-4.0), np.deg2rad(12.0)
n_pitch_frames = 120

In [ ]:
def pitch_states(attack_angles: np.ndarray) -> Iterator[tuple[Point, Point, Bivector, Force]]:
    """Yield flow states and lift as the wing pitches smoothly across `attack_angles`."""
    for angle in attack_angles:
        pitched_centre, pitched_trailing = wing_geometry(angle)                # [] Point-, Point-
        pitched_swirl = 4 * np.pi * (stream ^ (pitched_trailing - pitched_centre))   # [] Bivector+
        drawn_positions = cylinder_rings(pitched_centre, pitched_trailing, n_drawn_rings, n_drawn_angles, outer_distance)   # [n_drawn_rings, n_drawn_angles] Point-
        points, velocity, _, flux = wing_flow(pitched_centre, pitched_trailing, pitched_swirl, stream, drawn_positions, pitched_trailing)
        yield points, velocity, flux, -density * (pitched_swirl & stream)


attack_cycle = lowest_attack + (highest_attack - lowest_attack) * (1 - np.cos(np.linspace(0, 2 * np.pi, n_pitch_frames, endpoint=False))) / 2
pitched_states = show_states(
    pitch_states(attack_cycle), render.animate, speed,
    name="wing_pitch", duration_ms=frame_duration_ms, n_frames=n_pitch_frames,
);

## Contour circulation and surface pressure lift

Summing `velocity * step` around each closed ring integrates both flow fields once around the wing. Because the stream function is single-valued outside the wing, the bivector part sums to zero, confirming zero net mass flux across every ring. Because the velocity's geometric derivative vanishes in the annulus between any two rings, every ring yields the same circulation as `swirl`.

Along the wing boundary, Bernoulli pressure drops by half the density times `velocity.squared()`. Multiplying each boundary step by the unit intrinsic winding scalar `winding / winding.antinorm()` turns the step into the outward force covector per unit pressure, and summing around the surface recovers the Kutta–Joukowski lift across the stream with zero drag, known as d'Alembert's paradox.

In vector-calculus notation, the two parts of the loop integral read as the circulation $\oint \mathbf{v} \cdot d\mathbf{l}$ and the outward flux $\oint (\mathbf{v} \times d\mathbf{l}) \cdot \hat{\mathbf{z}}$, whose invariance across nested contours follows from Stokes's theorem and the divergence theorem.

In [ ]:
ring = Topology.loops([n_angles])
edge_average = ring.incidence[1, 0].mean(axis=0)                               # Scalar+[E <- V]
loop_points = wing_points.field()                                              # [n_rings] Point-[V]
loop_velocities = wing_velocity.field()                                        # [n_rings] Point-[V]
edge_steps = ring.d[0] * loop_points                                           # [n_rings] Point-[E]
ring_integrals = ((edge_average * loop_velocities) * edge_steps).batch().sum(axis=-1)   # [n_rings] Even+
tails, heads = ring.incidence[1, 0][:, None] * loop_points                     # [n_rings] Point-[E], Point-[E]
ring_areas = (tails ^ heads).batch().sum(axis=-1)                              # [n_rings] Bivector+
loop_circulations = ring_integrals * (ring_areas / ring_areas.norm())          # [n_rings] Even+
# Surface pressure force around the innermost ring (`wing_points[0]`).
speed_squared = wing_velocity[0].squared().field()                             # [] Scalar+[V]
edge_pressures = -0.5 * density * (edge_average * speed_squared)               # [] Scalar+[E]
winding = (tails[0] & heads[0]).batch().sum(axis=-1)                           # [] Antiscalar-
pressure_force = (winding / winding.antinorm()) * (edge_pressures * edge_steps[0]).batch().sum(axis=-1)   # [] Force+

In [ ]:
print(f"innermost ring circulation:\n{loop_circulations[0].coefficient_table()}")
print(f"outermost ring circulation:\n{loop_circulations[-1].coefficient_table()}")
print(f"Kutta circulation:\n{swirl.coefficient_table()}")
print(f"integrated pressure force:\n{pressure_force.coefficient_table()}")
print(f"Kutta-Joukowski lift:\n{lift.coefficient_table()}")

In [ ]:
# checks
# Stagnation at the cylinder trailing edge, and vanishing derivative past the cylinder.
trailing_velocity, _, _ = cylinder_flow(centre, trailing_edge, swirl, stream, trailing_edge)
assert (trailing_velocity.norm() < 1e-12).all()
assert (cylinder_derivatives.norm() < 1e-12).all()
# Critical points double under the Joukowski map, and Apollonius distance ratios square.
around_images = joukowski_image(trailing_edge, around_circles)                 # [6, n_outline_points] Point-
image_ratios = (around_images - 2 * trailing_edge).norm() / (around_images + 2 * trailing_edge).norm()   # [6, n_outline_points] Scalar+
assert ((joukowski_image(trailing_edge, trailing_edge) - 2 * trailing_edge).norm() < 1e-12).all()
assert ((image_ratios - (all_ratios**2)[:, None]).abs() < 1e-9 * (all_ratios**2)[:, None]).all()
# Stream function is constant along the wing surface, and morphing reaches the steady wing at midpoint.
assert ((wing_flux[0] - wing_flux[0, :1]).norm() < 1e-12).all()
_, _, midpoint_flux, _, midpoint_marks, _ = morphed_states[n_morph_frames // 2]
assert ((midpoint_marks[0] - 2 * trailing_edge).norm() < 1e-12).all()
assert ((midpoint_flux[0] - midpoint_flux[0, :1]).norm() < 1e-12).all()
# Pitching lift rises from nose-down minimum to nose-up peak and acts purely across the stream.
low_lift = pitched_states[0][3]                                                # [] Force+
high_lift = pitched_states[n_pitch_frames // 2][3]                             # [] Force+
assert ((high_lift & mv.y.dual()) > (low_lift & mv.y.dual())).all()
assert ((high_lift & stream).abs() < 1e-12).all()
# Loop circulation agrees across all rings with zero flux, and surface pressure recovers Kutta-Joukowski lift.
assert ((loop_circulations - loop_circulations[:1]).norm() < 1e-11).all()
assert ((loop_circulations - swirl).norm() < 1e-3).all()
assert ((pressure_force - lift).norm() < 1e-3).all()